In [2]:
import torch
import torch.nn as nn

In [3]:
class TimeEmbedding(nn.Module):
    # PE(t, 2i) = sin(t / 10000^(2i/d))
    # PE(t, 2i+1) = cos(t / 10000^(2i/d))
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        
    def forward(self, t):
        # t shape: (batch,)
        device = t.device
        half_dim = self.dim // 2
        i = torch.arange(0, half_dim, device=device) 
        div_term = 10000 ** (2 * i / self.dim)
        
        # t needs to be (batch, 1) to broadcast against div_term (half_dim,)
        t = t.unsqueeze(1).float()
        
        sins = torch.sin(t/div_term)
        coss = torch.cos(t/div_term)
        emb = torch.cat((sins, coss), dim=-1)
        return emb  # shape: (batch, dim)


In [4]:
# def residual_block(x, block): return x + block(x)
# def resblock(x, dim:int, in_channels:int , out_channels:int, embeddings:bool,
#               ks:int, padding:int=0, strides:int=1, num_groups:int=2):
#     block = nn.Sequential(
#         nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=ks,
#                   stride=strides, padding=padding),
#         nn.GroupNorm(num_groups, out_channels),
#         nn.SiLU(),
#         nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=ks,
#                           stride=strides, padding=padding),
#         nn.GroupNorm(num_groups, out_channels),
#         nn.SiLU()
#     )
#     # if embeddings:
#     #     embedding_layer = TimeEmbedding(dim=dim)
#     #     block.append(embedding_layer)
#     # downsample = nn.ConvTranspose2d(in_channels=out_channels, out_channels=out_channels*2,
#     #                                 kernel_size=ks, stride=strides, padding=padding)
#     # block.append(downsample).
#     return residual_block(x, block)
    
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, time_dim, num_groups=2):
        super().__init__()
        # block1: conv → groupnorm → silu
        self.block_1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, 
                      kernel_size=kernel_size, padding=1),
            nn.GroupNorm(num_groups, out_channels),
            nn.SiLU()
        )
        # block2: conv → groupnorm → silu
        self.block_2 = nn.Sequential(
            nn.Conv2d(in_channels=out_channels, out_channels=out_channels, 
                      kernel_size=kernel_size, padding=1),
            nn.GroupNorm(num_groups, out_channels),
            nn.SiLU()
        )
        # time_mlp: linear to project time embedding to out_channels
        self.mlp = nn.Linear(time_dim, out_channels)
        # residual_conv: 1x1 conv if in_channels != out_channels, else identity
        if in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_conv = nn.Identity()
        
    def forward(self, x, t):
        org_x = x
        # 1. pass x through block1
        x = self.block_1(x)
        # 2. project t embedding and add to output
        mapped_embeddings = self.mlp(t).unsqueeze(-1).unsqueeze(-1)    
        x = x+mapped_embeddings
        # 3. pass through block2
        x = self.block_2(x)
        # 4. add residual connection
        return self.residual_conv(org_x) + x

In [27]:
block = ResBlock(in_channels=1, out_channels=64, kernel_size=3, time_dim=128)
x = torch.randn(4, 1, 28, 28)
t_emb = TimeEmbedding(128)(torch.randint(0, 1000, (4,)))
out = block(x, t_emb)
print(out.shape)  # should be (4, 64, 24, 24)

torch.Size([4, 64, 28, 28])


In [5]:
class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(1, channels)
        self.attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        
    def forward(self, x):
        B, C, H, W = x.shape
        # normalize
        x_norm = self.norm(x)
        # reshape to sequence: (B, H*W, C)
        x_flat = x_norm.reshape(B, C, H*W).transpose(1, 2)
        # self attention
        attn_out, _ = self.attention(x_flat, x_flat, x_flat)
        # reshape back: (B, C, H, W)
        attn_out = attn_out.transpose(1, 2).reshape(B, C, H, W)
        # residual connection
        return x + attn_out

In [6]:
class Unet(nn.Module):
    def __init__(self, in_channels, out_channel, kernel_size, time_dim, num_groups:int=2):
        super().__init__()
        # Encoder Part ResBlocks
        self.encoder_resblock_1 = ResBlock(in_channels, out_channel, kernel_size, time_dim, num_groups)
        self.encoder_resblock_2 = ResBlock(out_channel, out_channel*2, kernel_size, time_dim, num_groups)
        # Encoder Downsamplers
        self.downsampler1 = nn.MaxPool2d(2)
        self.downsampler2 = nn.MaxPool2d(2)

        # BottleNeck:
        self.bottleneck = ResBlock(out_channel*2, out_channel*2, kernel_size, time_dim, num_groups)
        self.attn_ = SelfAttention(channels=out_channel*2)
        # Decoder Resblocks
        self.decoder_resblock_1 = ResBlock(out_channel*4, out_channel, kernel_size, time_dim, num_groups)
        self.decoder_resblock_2 = ResBlock(out_channel*2, out_channel, kernel_size, time_dim, num_groups)
        # Decoder Upsamplers
        self.upsampler1 = nn.Upsample(scale_factor=2)
        self.upsampler2 = nn.Upsample(scale_factor=2)

        # Embedding layer
        self.embeddings = TimeEmbedding(dim=time_dim)
        # output layer
        self.output_layer = nn.Conv2d(out_channel, in_channels, kernel_size=1)

    def forward(self, x, t):
        # 1. time embedding
        t_emb = self.embeddings(t)
        # 2. encoder block 1 + save skip1
        ex1 = self.encoder_resblock_1(x, t_emb)
        # 3. downsample
        down_ex1 = self.downsampler1(ex1)
        # 4. encoder block 2 + save skip2
        ex2 = self.encoder_resblock_2(down_ex1, t_emb)
        # 5. downsample
        down_ex2 = self.downsampler2(ex2)

        # 6. bottleneck
        btnx = self.bottleneck(down_ex2, t_emb)
        btnx = self.attn_(btnx)

        # 7. upsample
        up_dx1 = self.upsampler1(btnx)
        # 8. concat skip2 + decoder block 1
        cat1 = torch.cat([up_dx1, ex2], dim=1)
        dx1 = self.decoder_resblock_1(cat1, t_emb) 
        # 9. upsample
        up_dx2 = self.upsampler2(dx1)
        # 10. concat skip1 + decoder block 2
        cat2 = torch.cat([up_dx2, ex1], dim=1)
        dx2 = self.decoder_resblock_2(cat2, t_emb)

        # 11. output conv
        out = self.output_layer(dx2)
        return out

In [55]:
ds = nn.MaxPool2d(2)

In [56]:
ds(out).shape

torch.Size([4, 64, 14, 14])

In [57]:
block = ResBlock(in_channels=64, out_channels=128, kernel_size=3, time_dim=128)(ds(out), t_emb)

In [58]:
ds(block).shape

torch.Size([4, 128, 7, 7])

In [11]:
model = Unet(in_channels=3, out_channel=128, kernel_size=3, time_dim=128)
x = torch.randn(4, 3, 32, 32)
t = torch.randint(0, 1000, (4,))
out = model(x, t)
print(out.shape)  # should be (4, 1, 28, 28)

torch.Size([4, 3, 32, 32])


In [12]:
total_params = sum(p.numel() for p in model.parameters())

In [13]:
total_params

3912067